## Import  Packets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!git clone https://github.com/UBC-NLP/VioletV2.git

Cloning into 'VioletV2'...
remote: Enumerating objects: 285, done.
remote: Counting objects: 100% (285/285), done.
remote: Compressing objects: 100% (190/190), done.
remote: Total 285 (delta 151), reused 193 (delta 89), pack-reused 0 (from 0)
Receiving objects: 100% (285/285), 6.34 MiB | 21.37 MiB/s, done.
Resolving deltas: 100% (151/151), done.


In [ ]:
import sys
sys.path.append('/content/VioletV2')

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import torch
from models.transformer import Violet, VisualEncoder, ScaledDotProductAttention
from transformers import AutoTokenizer
import h5py
import json
from collections import defaultdict
from torch.utils.data import DataLoader
from dataset_refactored import Coco_Dataset

from evaluation.cider.cider import Cider
from evaluation.bleu.bleu import Bleu
from evaluation.meteor.meteor import Meteor
from evaluation.rouge.rouge import Rouge
from tqdm import tqdm




---



## Checkpoint

Early checkpoint: [Download](https://mbzuaiac-my.sharepoint.com/:u:/g/personal/abdelrahman_mohamed_mbzuai_ac_ae/Ed04OUJoG4tLtUipJRWW4fgBPs2qUfVAIQUt_Aym40W-Aw?e=3S2E01)

In [ ]:
checkpoint_path = "/content/drive/MyDrive/Colab Notebooks/Violet Model 2/Violet_checkpoint_0.pth"

In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=torch.device('cuda'))

In [ ]:
checkpoint.keys()

dict_keys(['torch_rng_state', 'cuda_rng_state', 'numpy_rng_state', 'random_rng_state', 'epoch', 'val_loss', 'val_cider', 'state_dict', 'optimizer', 'patience', 'best_cider', 'use_rl'])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("UBC-NLP/Jasmine-350M")
encoder = VisualEncoder(N=3, padding_idx=0, attention_module=ScaledDotProductAttention)
model = Violet(bos_idx=tokenizer.vocab['<|endoftext|>'], encoder=encoder, n_layer=12, tau=0.3)

tokenizer_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.51M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.56M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.65M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.52k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/593M [00:00<?, ?B/s]

In [ ]:
model.load_state_dict(checkpoint['state_dict'], strict=False)
model.eval()
model = model.to("cuda")

In [ ]:
model

Violet(
  (encoder): VisualEncoder(
    (layers): ModuleList(
      (0-2): 3 x EncoderLayer(
        (mhatt): MultiHeadAttention(
          (attention): ScaledDotProductAttention(
            (fc_q): Linear(in_features=768, out_features=768, bias=True)
            (fc_k): Linear(in_features=768, out_features=768, bias=True)
            (fc_v): Linear(in_features=768, out_features=768, bias=True)
            (fc_o): Linear(in_features=768, out_features=768, bias=True)
            (c_attn_query): Conv1D()
            (c_attn_key): Conv1D()
            (c_attn_value): Conv1D()
          )
          (dropout): Dropout(p=0.1, inplace=False)
          (layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        )
        (pwff): PositionWiseFeedForward(
          (fc1): Linear(in_features=768, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (dropout_2): Dropout(p=0



---



## DataSet

Coco Images HDF5 file: [Download](https://mbzuaiac-my.sharepoint.com/:u:/g/personal/abdelrahman_mohamed_mbzuai_ac_ae/EZUDaVbRzGFJuYbYnU_jZ0YBnjZgPSuG32Z6wlLeCT22iQ?e=fI1rG0)

Annotations: [Download](https://mbzuaiac-my.sharepoint.com/:u:/g/personal/abdelrahman_mohamed_mbzuai_ac_ae/EXkwLG9hEE5EimCVLsVpHTwB6EadaXDCXBII3lBquptmjw?e=fD24oa)

In [ ]:
images_path = "/content/drive/MyDrive/Colab Notebooks/Violet Model 2/coco_images.h5"
annotations_path = "/content/drive/MyDrive/Colab Notebooks/Violet Model 2/annotations/"

In [ ]:
dataset_val = Coco_Dataset(img_root=images_path, ann_root=annotations_path,split="val")
dataloader_val = DataLoader(dataset_val, batch_size=32, shuffle=False, num_workers=4, collate_fn=dataset_val.collate_fn)


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/961k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [ ]:
len(dataset_val)

185866

In [ ]:
dataset_val[0]

(tensor([[[1.9303, 1.9303, 1.9303,  ..., 1.9303, 1.9303, 1.9303],
          [1.9303, 1.9303, 1.9303,  ..., 1.9303, 1.9303, 1.9303],
          [1.9303, 1.9303, 1.9303,  ..., 1.9303, 1.9303, 1.9303],
          ...,
          [1.9303, 1.9303, 1.9303,  ..., 1.9303, 1.9303, 1.9303],
          [1.9303, 1.9303, 1.9303,  ..., 1.9303, 1.9303, 1.9303],
          [1.9303, 1.9303, 1.9303,  ..., 1.9303, 1.9303, 1.9303]],
 
         [[2.0749, 2.0749, 2.0749,  ..., 2.0749, 2.0749, 2.0749],
          [2.0749, 2.0749, 2.0749,  ..., 2.0749, 2.0749, 2.0749],
          [2.0749, 2.0749, 2.0749,  ..., 2.0749, 2.0749, 2.0749],
          ...,
          [2.0749, 2.0749, 2.0749,  ..., 2.0749, 2.0749, 2.0749],
          [2.0749, 2.0749, 2.0749,  ..., 2.0749, 2.0749, 2.0749],
          [2.0749, 2.0749, 2.0749,  ..., 2.0749, 2.0749, 2.0749]],
 
         [[2.1459, 2.1459, 2.1459,  ..., 2.1459, 2.1459, 2.1459],
          [2.1459, 2.1459, 2.1459,  ..., 2.1459, 2.1459, 2.1459],
          [2.1459, 2.1459, 2.1459,  ...,



---



## Evaluate Model

In [ ]:
def load_references_from_json(file_path):

    with open(file_path, 'r') as f:
        data = json.load(f)

    captions_by_id = defaultdict(list)
    for item in data["annotations"]:
        captions_by_id[item['image_id']].append(item['caption'])

    return captions_by_id

def calculate_metrics(generated_captions, ref_caps):
    cider_scorer = Cider()
    bleu_scorer = Bleu(4)
    meteor_scorer = Meteor()
    rouge_scorer = Rouge()

    cider_score, _ = cider_scorer.compute_score(ref_caps, generated_captions)
    bleu_score, _ = bleu_scorer.compute_score(ref_caps, generated_captions)
    meteor_score, _ = meteor_scorer.compute_score(ref_caps, generated_captions)
    rouge_score, _ = rouge_scorer.compute_score(ref_caps, generated_captions)


    metrics = {
        'CIDEr': cider_score,
        'BLEU-1': bleu_score[0],
        'BLEU-2': bleu_score[1],
        'BLEU-3': bleu_score[2],
        'BLEU-4': bleu_score[3],
        'METEOR': meteor_score,
        'ROUGE': rouge_score
    }

    return metrics

def evaluation(model, dataloader_val, ref_caps):
    tokenizer = AutoTokenizer.from_pretrained("UBC-NLP/Jasmine-350M")
    model.eval()
    model = model.to("cuda")
    gen_caps = {}
    with tqdm( unit='it', total=len(dataloader_val)) as pbar:
        for it, (images, captions, ids) in enumerate(dataloader_val):
            images, captions = images.to("cuda"), captions.to("cuda")

            with torch.no_grad():
                out, _ = model.beam_search(images, 40, tokenizer.vocab['<|endoftext|>'], 5, out_size=1)
                generated_caption = tokenizer.batch_decode(out, skip_special_tokens=True)
                output = {key: [value] for key,value in zip(ids[0], generated_caption)}

            gen_caps = {**gen_caps, **output}
            pbar.update()
    ref_caps = dict(list(ref_caps.items())[0:len(gen_caps)])
    metrics = calculate_metrics(gen_caps, ref_caps)
    return metrics

In [ ]:
ref_caps = load_references_from_json("/content/drive/MyDrive/Colab Notebooks/Violet Model 2/annotations/NLLB_val_coco.json")
metrics = evaluation(model, dataloader_val, ref_caps)

100%|██████████| 5809/5809 [2:57:15<00:00,  1.83s/it]


In [ ]:
pd.DataFrame([metrics])

,CIDEr,BLEU-1,BLEU-2,BLEU-3,BLEU-4,METEOR,ROUGE
0,0.98777,0.650394,0.508735,0.39389,0.306021,0.364682,0.481619


In [ ]:
json.dump(metrics, open("/content/drive/MyDrive/Colab Notebooks/Violet Model 2/metrics.json", "w"))



---



## Predict Caption

In [ ]:
def predict_caption(model, image_tensor, max_length=40, beam_size=5):
    model.eval()
    tokenizer = AutoTokenizer.from_pretrained("UBC-NLP/Jasmine-350M")
    with torch.no_grad():
        output, _ = model.beam_search(image_tensor, max_length, tokenizer.vocab['<|endoftext|>'], beam_size, out_size=1)
        generated_caption = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
    return generated_caption

In [ ]:
image_tensor = dataset_val[0][0].unsqueeze(0).to("cuda")
caption = predict_caption(model, image_tensor)
print(f"Generated Caption: {caption}")

Generated Caption:  الدراجة لديها ساعة مربوطة بها




---

